# 03 — Análise em SQL

As mesmas perguntas dos gráficos, respondidas em **SQL** sobre um banco SQLite.

- `src/banco.py` monta `data/ufc.db` a partir dos CSVs limpos, seguindo `sql/schema.sql`: os CSVs (uma linha por lutador em cada luta) viram cinco tabelas — `eventos` (com a data de cada evento) → `lutas` → `desempenho` → `desempenho_round`, mais `lutadores` (altura, envergadura, base, nascimento) ligada a `desempenho`.
- As consultas ficam em `sql/consultas.sql`, cada uma com um `-- nome:`; `src/consultas.py` as lê e executa.
- Cada resposta é **conferida com Pandas**: se os dois caminhos não baterem, a célula falha.

In [1]:
import sys
sys.path.append("..")

import sqlite3
import pandas as pd

from src.banco import criar_banco, ARQUIVO_BANCO
from src.consultas import carregar_consultas, consultar

# Recria o banco a partir dos CSVs (rápido; garante que está em dia com os dados)
for tabela, linhas in criar_banco().items():
    print(f"{tabela}: {linhas} linhas")

consultas = carregar_consultas()
df = pd.read_csv("../data/processed/dataset_final_resumo_limpo.csv")  # para conferência


def mostrar(nome):
    """Imprime o SQL da consulta e devolve o resultado."""
    print(consultas[nome], end="\n\n")
    return consultar(nome)

eventos: 149 linhas
lutas: 1837 linhas
lutadores: 964 linhas
desempenho: 3674 linhas
desempenho_round: 8910 linhas


## O modelo

Tabelas e quantidade de linhas. A chave de `desempenho` é `(luta_id, lutador)`; cada luta tem exatamente dois registros.

In [2]:
with sqlite3.connect(ARQUIVO_BANCO) as conexao:
    tabelas = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY rowid", conexao)["name"]
    print(pd.DataFrame({t: [pd.read_sql_query(f"SELECT COUNT(*) FROM {t}", conexao).iloc[0, 0]] for t in tabelas}, index=["linhas"]).T)
    print("Lutas com número de lutadores diferente de 2:",
          pd.read_sql_query("SELECT COUNT(*) FROM (SELECT luta_id FROM desempenho GROUP BY luta_id HAVING COUNT(*) <> 2)", conexao).iloc[0, 0])

                  linhas
eventos              149
lutas               1837
lutadores            964
desempenho          3674
desempenho_round    8910
Lutas com número de lutadores diferente de 2: 0


## 1. Como as lutas terminam

In [3]:
metodos = mostrar("metodos_vitoria")
metodos

-- Como as lutas terminam? Uma linha por luta, então não há contagem dobrada.
SELECT
    metodo,
    COUNT(*)                                            AS lutas,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)  AS pct
FROM lutas
GROUP BY metodo
ORDER BY lutas DESC;



,metodo,lutas,pct
0,Decision - Unanimous,731,39.8
1,KO/TKO,588,32.0
2,Submission,313,17.0
3,Decision - Split,158,8.6
4,Decision - Majority,15,0.8
5,Could Not Continue,13,0.7
6,TKO - Doctor's Stoppage,9,0.5
7,Overturned,7,0.4
8,DQ,3,0.2


In [4]:
# Conferência: mesma contagem do gráfico metodos_vitoria.png (uma linha por luta)
esperado = df.drop_duplicates("Fight_URL")["Method"].value_counts()
assert metodos.set_index("metodo")["lutas"].to_dict() == esperado.to_dict()
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 2. O que separa vencedores de perdedores

In [5]:
vp = mostrar("vencedores_perdedores")
vp

-- O que separa quem vence de quem perde? Médias por luta de cada lado.
WITH medias AS (
    SELECT
        resultado,
        AVG(sig_acertados)            AS golpes_sig,
        AVG(quedas)                   AS quedas,
        AVG(controle_seg) / 60.0      AS controle_min,
        AVG(tent_finalizacao)         AS tent_finalizacao,
        AVG(kd)                       AS knockdowns
    FROM desempenho
    WHERE resultado IN ('W', 'L')
    GROUP BY resultado
)
SELECT
    ROUND(v.golpes_sig, 1)                         AS golpes_sig_v,
    ROUND(p.golpes_sig, 1)                         AS golpes_sig_p,
    ROUND(v.golpes_sig / p.golpes_sig, 1)          AS golpes_sig_razao,
    ROUND(v.quedas / p.quedas, 1)                  AS quedas_razao,
    ROUND(v.controle_min / p.controle_min, 1)      AS controle_razao,
    ROUND(v.tent_finalizacao / p.tent_finalizacao, 1) AS finalizacao_razao,
    ROUND(v.knockdowns / p.knockdowns, 1)          AS knockdowns_razao
FROM medias v
CROSS JOIN medias p


,golpes_sig_v,golpes_sig_p,golpes_sig_razao,quedas_razao,controle_razao,finalizacao_razao,knockdowns_razao
0,52.0,35.4,1.5,2.1,2.1,2.6,8.4


In [6]:
medias = df[df["Resultado"].isin(["W", "L"])].groupby("Resultado")[["Sig_str_landed", "KD", "Td_landed"]].mean()
assert round(medias.loc["W", "KD"] / medias.loc["L", "KD"], 1) == vp.loc[0, "knockdowns_razao"]
assert round(medias.loc["W", "Td_landed"] / medias.loc["L", "Td_landed"], 1) == vp.loc[0, "quedas_razao"]
assert round(medias.loc["W", "Sig_str_landed"], 1) == vp.loc[0, "golpes_sig_v"]
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 3. Taxa de vitória (mín. 6 lutas)

`RANK()` deixa explícitos os empates: vários invictos dividem a 1ª posição.

In [7]:
taxa = mostrar("taxa_vitoria")
taxa

-- Top 10 por taxa de vitória, com amostra mínima de 6 lutas (NC não entra na conta).
-- RANK() mostra os empates: vários invictos dividem a 1ª posição.
SELECT
    RANK() OVER (ORDER BY 1.0 * SUM(resultado = 'W') / COUNT(*) DESC) AS posicao,
    lutador,
    SUM(resultado = 'W')                                  AS vitorias,
    SUM(resultado = 'L')                                  AS derrotas,
    SUM(resultado = 'D')                                  AS empates,
    ROUND(100.0 * SUM(resultado = 'W') / COUNT(*), 1)     AS taxa_pct
FROM desempenho
WHERE resultado IN ('W', 'L', 'D')
GROUP BY lutador
HAVING COUNT(*) >= 6
ORDER BY taxa_pct DESC, vitorias DESC
LIMIT 10;



,posicao,lutador,vitorias,derrotas,empates,taxa_pct
0,1,Carlos Ulberg,7,0,0,100.0
1,1,Ailin Perez,6,0,0,100.0
2,1,Farid Basharat,6,0,0,100.0
3,1,Natalia Silva,6,0,0,100.0
4,1,Navajo Stirling,6,0,0,100.0
5,1,Quillan Salkilld,6,0,0,100.0
6,7,Joshua Van,11,1,0,91.7
7,8,Asu Almabayev,7,1,0,87.5
8,8,Carlos Prates,7,1,0,87.5
9,8,Denise Gomes,7,1,0,87.5


In [8]:
cartel = pd.crosstab(df["Fighter"], df["Resultado"]).reindex(columns=["W", "L", "D"], fill_value=0)
cartel["lutas"] = cartel.sum(axis=1)
cartel = cartel[cartel["lutas"] >= 6]
cartel["taxa"] = cartel["W"] / cartel["lutas"] * 100
esperado = cartel.sort_values(["taxa", "W"], ascending=False).head(10)
assert list(taxa["lutador"]) == list(esperado.index)
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 4. Precisão de quedas (mín. 15 tentativas e 3 lutas)

In [9]:
quedas = mostrar("precisao_quedas")
quedas

-- Top 10 por precisão de quedas: pelo menos 15 tentativas E 3 lutas no período.
SELECT
    lutador,
    SUM(quedas)                                           AS conseguidas,
    SUM(quedas_tentadas)                                  AS tentadas,
    COUNT(*)                                              AS lutas,
    ROUND(100.0 * SUM(quedas) / SUM(quedas_tentadas), 1)  AS precisao_pct
FROM desempenho
GROUP BY lutador
HAVING SUM(quedas_tentadas) >= 15 AND COUNT(*) >= 3
ORDER BY precisao_pct DESC, conseguidas DESC
LIMIT 10;



,lutador,conseguidas,tentadas,lutas,precisao_pct
0,Mason Jones,11,15,4,73.3
1,Andre Muniz,14,20,4,70.0
2,Fatima Kline,14,21,5,66.7
3,Joanderson Brito,14,21,8,66.7
4,Cory Sandhagen,10,15,5,66.7
5,Zhang Weili,13,20,4,65.0
6,Joaquin Buckley,12,19,8,63.2
7,Hamdy Abdelwahab,10,16,3,62.5
8,Lerone Murphy,10,16,6,62.5
9,Piera Rodriguez,18,29,5,62.1


In [10]:
g = df.groupby("Fighter").agg(c=("Td_landed", "sum"), t=("Td_attempted", "sum"), n=("Fight_URL", "nunique"))
g = g[(g["t"] >= 15) & (g["n"] >= 3)]
g["p"] = g["c"] / g["t"] * 100
esperado = g.sort_values(["p", "c"], ascending=False).head(10)
assert list(quedas["lutador"]) == list(esperado.index)
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 5. Em que round terminam nocautes e finalizações

Percentual **dentro de cada método** com `SUM(COUNT(*)) OVER (PARTITION BY tipo)`.

In [11]:
rounds = mostrar("round_final_por_metodo")
rounds

-- Em que round terminam nocautes e finalizações? Percentual dentro de cada método
-- (função de janela particionada por método). Rounds 4 e 5 agrupados.
WITH finalizadas AS (
    SELECT
        CASE
            WHEN metodo LIKE '%KO/TKO%' OR metodo LIKE 'TKO - Doctor%' THEN 'Nocaute'
            WHEN metodo LIKE '%Submission%' THEN 'Finalização'
        END AS tipo,
        CASE WHEN round_final >= 4 THEN '4º ou 5º' ELSE round_final || 'º' END AS round_grupo
    FROM lutas
)
SELECT
    tipo,
    round_grupo,
    COUNT(*)                                                               AS lutas,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY tipo), 1)    AS pct_do_metodo
FROM finalizadas
WHERE tipo IS NOT NULL
GROUP BY tipo, round_grupo
ORDER BY tipo DESC, round_grupo;



,tipo,round_grupo,lutas,pct_do_metodo
0,Nocaute,1º,320,53.6
1,Nocaute,2º,176,29.5
2,Nocaute,3º,90,15.1
3,Nocaute,4º ou 5º,11,1.8
4,Finalização,1º,144,46.0
5,Finalização,2º,109,34.8
6,Finalização,3º,56,17.9
7,Finalização,4º ou 5º,4,1.3


In [12]:
lutas_df = df.drop_duplicates("Fight_URL")
for tipo, padrao in [("Nocaute", "KO/TKO|TKO - Doctor"), ("Finalização", "Submission")]:
    do_metodo = lutas_df[lutas_df["Method"].str.contains(padrao, na=False)]
    esperado = round((do_metodo["Final_Round"] == 1).mean() * 100, 1)
    assert rounds.query("tipo == @tipo and round_grupo == '1º'")["pct_do_metodo"].item() == esperado
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem

## 6. De onde saem os golpes de vencedores e perdedores

In [13]:
origem = mostrar("origem_golpes")
origem

-- De onde saem os golpes de vencedores e perdedores (distância, clinch, chão)?
SELECT
    CASE resultado WHEN 'W' THEN 'Vencedores' ELSE 'Perdedores' END  AS grupo,
    ROUND(100.0 * SUM(distancia) / SUM(sig_acertados), 1)            AS distancia_pct,
    ROUND(100.0 * SUM(clinch)    / SUM(sig_acertados), 1)            AS clinch_pct,
    ROUND(100.0 * SUM(chao)      / SUM(sig_acertados), 1)            AS chao_pct
FROM desempenho
WHERE resultado IN ('W', 'L')
GROUP BY resultado
ORDER BY grupo DESC;



,grupo,distancia_pct,clinch_pct,chao_pct
0,Vencedores,77.3,9.5,13.2
1,Perdedores,86.8,9.7,3.5


In [14]:
wl = df[df["Resultado"].isin(["W", "L"])].groupby("Resultado")[["Ground_landed", "Sig_str_landed"]].sum()
chao = (wl["Ground_landed"] / wl["Sig_str_landed"] * 100).round(1)
assert origem.set_index("grupo").loc["Vencedores", "chao_pct"] == chao["W"]
assert origem.set_index("grupo").loc["Perdedores", "chao_pct"] == chao["L"]
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 7. Quem mais arrisca golpes (por minuto de luta)

`JOIN` com `lutas` para usar a duração real de cada luta.

In [15]:
volume = mostrar("volume_por_minuto")
volume

-- Quem mais arrisca golpes? Tentativas por minuto de luta (JOIN com lutas para a duração),
-- lutadores com pelo menos 4 lutas.
SELECT
    d.lutador,
    COUNT(*)                                                       AS lutas,
    ROUND(SUM(d.sig_tentados) / (SUM(l.duracao_seg) / 60.0), 1)    AS tentados_por_min,
    ROUND(100.0 * SUM(d.sig_acertados) / SUM(d.sig_tentados), 1)   AS precisao_pct
FROM desempenho AS d
JOIN lutas AS l USING (luta_id)
GROUP BY d.lutador
HAVING COUNT(*) >= 4
ORDER BY tentados_por_min DESC
LIMIT 10;



,lutador,lutas,tentados_por_min,precisao_pct
0,Carlos Leal,4,18.6,47.0
1,Esteban Ribovics,7,18.6,43.1
2,Nasrat Haqparast,6,18.3,46.7
3,Carli Judice,5,18.1,48.1
4,Terrance McKinney,9,17.4,59.4
5,Djorden Santos,4,16.9,39.5
6,Mason Jones,4,16.6,43.1
7,Josh Hokit,4,16.1,63.8
8,Diana Belbita,4,15.7,41.3
9,Davey Grant,5,15.3,46.9


## 8. Árbitros

O árbitro não escolhe o método: a taxa reflete as lutas que cada um recebe.

In [16]:
arbitros = mostrar("arbitros")
arbitros

-- Os 10 árbitros com mais lutas e a % delas encerrada antes da decisão.
SELECT
    arbitro,
    COUNT(*)                                                                    AS lutas,
    ROUND(100.0 * SUM(metodo LIKE '%KO/TKO%' OR metodo LIKE 'TKO - Doctor%'
                      OR metodo LIKE '%Submission%') / COUNT(*), 1)             AS pct_antes_da_decisao
FROM lutas
GROUP BY arbitro
ORDER BY lutas DESC
LIMIT 10;



,arbitro,lutas,pct_antes_da_decisao
0,Herb Dean,212,55.2
1,Jason Herzog,197,47.2
2,Marc Goddard,194,51.0
3,Chris Tognoni,167,44.9
4,Mark Smith,159,51.6
5,Mike Beltran,117,53.0
6,Keith Peterson,113,39.8
7,Kerry Hatley,90,55.6
8,Dan Miragliotta,76,46.1
9,Rich Mitchell,36,41.7


## 9. Golpes por round

Tende a subir por viés de sobrevivência: só lutas longas chegam aos rounds finais.

In [17]:
mostrar("golpes_por_round")

-- Média de golpes significativos acertados por round (tende a subir por viés de sobrevivência:
-- só lutas longas chegam aos rounds finais).
SELECT
    round,
    COUNT(*)                     AS registros,
    ROUND(AVG(sig_acertados), 1) AS media_golpes_sig
FROM desempenho_round
GROUP BY round
ORDER BY round;



,round,registros,media_golpes_sig
0,1,3674,16.2
1,2,2724,18.9
2,3,2138,19.7
3,4,196,19.6
4,5,178,20.5


## 10. Canhoto × ortodoxo

*Self-join* de `desempenho`: a linha do vencedor encontra a do perdedor da mesma luta, e cada uma é ligada à base do lutador em `lutadores`. Os confrontos vêm de uma lista em `VALUES`, que fixa a ordem e qual lado é contado.

In [18]:
bases = mostrar("confronto_bases")
bases

-- Canhoto x ortodoxo: em lutas com vencedor entre bases diferentes, quantas o 1º lado venceu.
-- Self-join de desempenho (linha do vencedor com a do perdedor da mesma luta), cada uma ligada
-- à base do lutador em lutadores. Os confrontos são listados em VALUES para fixar a ordem e o lado.
WITH duelos AS (
    SELECT lv.base AS base_vencedor, lp.base AS base_perdedor
    FROM desempenho AS v
    JOIN desempenho AS p  ON p.luta_id = v.luta_id AND p.resultado = 'L'
    JOIN lutadores  AS lv ON lv.url = v.lutador_url
    JOIN lutadores  AS lp ON lp.url = p.lutador_url
    WHERE v.resultado = 'W' AND lv.base <> lp.base  -- base NULL fica de fora pela comparação
),
confrontos (ordem, lado_a, lado_b) AS (
    VALUES (1, 'Southpaw', 'Orthodox'), (2, 'Switch', 'Orthodox'), (3, 'Switch', 'Southpaw')
)
SELECT
    c.lado_a || ' x ' || c.lado_b                                         AS confronto,
    COUNT(*)                                                              AS lutas,
    SUM(d.base_v

,confronto,lutas,vitorias_lado_a,pct_lado_a
0,Southpaw x Orthodox,465,245,52.7
1,Switch x Orthodox,267,139,52.1
2,Switch x Southpaw,55,34,61.8


In [19]:
# Conferência: mesma contagem do gráfico base_canhoto_vs_ortodoxo.png, com o merge por Fighter_URL em Pandas
lutadores = pd.read_csv("../data/processed/lutadores.csv")
com_base = df.merge(lutadores[["Fighter_URL", "Stance"]], on="Fighter_URL")
duelos = pd.concat([
    com_base[com_base["Resultado"] == "W"].set_index("Fight_URL")["Stance"].rename("v"),
    com_base[com_base["Resultado"] == "L"].set_index("Fight_URL")["Stance"].rename("p"),
], axis=1, join="inner").dropna()
for linha in bases.itertuples():
    a, b = linha.confronto.split(" x ")
    no_confronto = ((duelos["v"] == a) & (duelos["p"] == b)) | ((duelos["v"] == b) & (duelos["p"] == a))
    assert (linha.lutas, linha.vitorias_lado_a) == (no_confronto.sum(), (no_confronto & (duelos["v"] == a)).sum())
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem


## 11. Disputas de cinturão × demais lutas

A CTE soma os dois lutadores de cada luta; depois, um `GROUP BY` pela flag `disputa_titulo`. Ritmo por minuto e por lutador, porque as disputas são de 5 rounds.

In [20]:
titulo = mostrar("disputas_titulo")
titulo

-- Disputas de cinturão x demais lutas. A CTE soma os dois lutadores de cada luta; o ritmo é por
-- minuto de luta e por lutador (duração x 2), porque as disputas têm 5 rounds e duram mais.
WITH por_luta AS (
    SELECT l.luta_id, l.disputa_titulo, l.duracao_seg, l.metodo, l.bonus_luta, l.bonus_performance,
           SUM(d.sig_acertados) AS golpes,
           SUM(d.kd)            AS kd
    FROM lutas AS l
    JOIN desempenho AS d USING (luta_id)
    GROUP BY l.luta_id
)
SELECT
    CASE disputa_titulo WHEN 1 THEN 'Disputa de cinturão' ELSE 'Demais lutas' END  AS tipo,
    COUNT(*)                                                                      AS lutas,
    ROUND(AVG(duracao_seg) / 60.0, 1)                                             AS duracao_min,
    ROUND(SUM(golpes) / (SUM(duracao_seg) / 60.0 * 2), 2)                         AS golpes_por_min,
    ROUND(SUM(kd) * 15 / (SUM(duracao_seg) / 60.0 * 2), 2)                        AS kd_por_15min,
    ROUND(100.0 * AVG(metodo LIKE '

,tipo,lutas,duracao_min,golpes_por_min,kd_por_15min,nocaute_pct,finalizacao_pct,decisao_pct,luta_da_noite_pct,performance_pct
0,Disputa de cinturão,76,16.8,3.96,0.21,34.2,11.8,52.6,18.4,31.6
1,Demais lutas,1761,10.7,3.97,0.31,32.4,17.3,49.1,4.9,21.8


In [21]:
# Conferência: mesmos números do gráfico disputas_de_cinturao.png, calculados em Pandas
minutos = df["Time"].str.split(":", expand=True).astype(int)
por_luta = df.assign(dur=(df["Final_Round"] - 1) * 300 + minutos[0] * 60 + minutos[1]).groupby("Fight_URL").agg(
    titulo=("Title_Bout", "first"), dur=("dur", "first"), golpes=("Sig_str_landed", "sum"), bonus=("Fight_Bonus", "first"))
g = por_luta.groupby("titulo").agg(lutas=("dur", "size"), dur=("dur", "sum"), golpes=("golpes", "sum"), bonus=("bonus", "mean"))
sql = titulo.assign(titulo=(titulo["tipo"] == "Disputa de cinturão").astype(int)).set_index("titulo")
g = g.reindex(sql.index)  # mesma ordem do SQL (disputas primeiro)
assert (sql["lutas"] == g["lutas"]).all()
assert (sql["golpes_por_min"] == (g["golpes"] / (g["dur"] / 60 * 2)).round(2)).all()
assert (sql["luta_da_noite_pct"] == (g["bonus"] * 100).round(1)).all()
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem

## 12. Finalizações por categoria de peso

`HAVING` aplica a amostra mínima (20 lutas) depois do agrupamento, `RANK()` numera as categorias e uma subconsulta escalar traz a média geral para cada linha.

In [22]:
categorias = mostrar("finalizacoes_por_categoria")
categorias

-- Lutas encerradas antes da decisão, por categoria de peso: nocaute (inclui TKO médico) e
-- finalização. Fora o peso casado (acordo entre os lutadores, não é divisão) e, via HAVING,
-- categorias com menos de 20 lutas. media_geral: todas as lutas fora o peso casado.
WITH por_categoria AS (
    SELECT
        categoria,
        COUNT(*)                                                               AS lutas,
        100.0 * AVG(metodo LIKE '%KO/TKO%' OR metodo LIKE 'TKO - Doctor%')     AS nocaute_pct,
        100.0 * AVG(metodo LIKE '%Submission%')                                AS finalizacao_pct
    FROM lutas
    WHERE categoria <> 'Catch Weight'
    GROUP BY categoria
    HAVING COUNT(*) >= 20
)
SELECT
    RANK() OVER (ORDER BY nocaute_pct + finalizacao_pct DESC)  AS posicao,
    categoria,
    lutas,
    ROUND(nocaute_pct, 1)                                      AS nocaute_pct,
    ROUND(finalizacao_pct, 1)                                  AS finalizacao_pct,
    ROUND(nocaute_pct

,posicao,categoria,lutas,nocaute_pct,finalizacao_pct,antes_da_decisao_pct,media_geral
0,1,Light Heavyweight,127,51.2,16.5,67.7,49.8
1,2,Middleweight,230,39.6,17.4,57.0,49.8
2,3,Lightweight,250,38.8,17.6,56.4,49.8
3,4,Featherweight,224,38.4,16.5,54.9,49.8
4,5,Heavyweight,129,41.9,11.6,53.5,49.8
5,6,Welterweight,211,37.0,16.1,53.1,49.8
6,7,Flyweight,141,23.4,21.3,44.7,49.8
7,8,Bantamweight,205,21.0,16.1,37.1,49.8
8,9,Women's Bantamweight,71,14.1,22.5,36.6,49.8
9,10,Women's Strawweight,126,17.5,18.3,35.7,49.8


In [23]:
# Conferência: mesma conta do gráfico finalizacoes_por_categoria.png (uma linha por luta, sem peso casado)
lutas_cat = df.drop_duplicates("Fight_URL")
lutas_cat = lutas_cat[lutas_cat["Weight_Class"] != "Catch Weight"].assign(
    nocaute=lambda x: x["Method"].str.contains("KO/TKO|TKO - Doctor", na=False),
    finalizacao=lambda x: x["Method"].str.contains("Submission", na=False))
g = lutas_cat.groupby("Weight_Class").agg(lutas=("Fight_URL", "size"), nocaute=("nocaute", "mean"), finalizacao=("finalizacao", "mean"))
g = g[g["lutas"] >= 20]
sql = categorias.set_index("categoria")
assert set(sql.index) == set(g.index)
g = g.reindex(sql.index)  # mesma ordem do SQL (por posição)
assert (sql["lutas"] == g["lutas"]).all()
assert (sql["antes_da_decisao_pct"] == ((g["nocaute"] + g["finalizacao"]) * 100).round(1)).all()
assert sql["media_geral"].iloc[0] == round((lutas_cat["nocaute"] | lutas_cat["finalizacao"]).mean() * 100, 1)
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem

## 13. Idade × envergadura

Idade na data do evento com `julianday()` (diferença em dias ÷ 365,25). *Self-join* do vencedor com o perdedor de cada luta, `CASE` para a faixa de diferença e `AVG` da condição (0/1) para a taxa. A linha `todas` vem de um `UNION ALL` que repete as lutas com a faixa 0.

In [24]:
fisico = mostrar("idade_envergadura")
fisico

-- Em lutas com vencedor, quem tinha a vantagem venceu com que frequência? Idade na data do
-- evento com julianday(); envergadura em polegadas (1" = 2,54 cm). Self-join do vencedor com o
-- perdedor da mesma luta; cada diferença cai numa faixa (CASE) e AVG da condição dá a taxa.
-- ordem 0 = todas as lutas do painel. Diferença zero fica de fora.
WITH fisico AS (
    SELECT
        d.luta_id,
        d.resultado,
        (julianday(e.data) - julianday(f.nascimento)) / 365.25  AS idade,
        ROUND(f.envergadura_cm / 2.54, 1)                       AS envergadura_pol
    FROM desempenho AS d
    JOIN lutas      AS l USING (luta_id)
    JOIN eventos    AS e USING (evento_id)
    JOIN lutadores  AS f ON f.url = d.lutador_url
    WHERE d.resultado IN ('W', 'L')
),
duelos AS (
    SELECT
        v.idade - p.idade                                       AS dif_idade,  -- negativo = vencedor mais novo
        ROUND(v.envergadura_pol - p.envergadura_pol, 1)         AS dif_env     -- positivo = 

,vantagem,ordem,diferenca,lutas,pct_vantagem_venceu
0,Mais novo,0,todas,1808,60.3
1,Mais novo,1,até 2 anos,545,54.3
2,Mais novo,2,2 a 5 anos,606,57.8
3,Mais novo,3,5 a 8 anos,406,64.3
4,Mais novo,4,mais de 8 anos,251,73.3
5,Mais envergadura,0,todas,1572,51.2
6,Mais envergadura,1,"1""",435,49.7
7,Mais envergadura,2,"2""",349,48.1
8,Mais envergadura,3,"3""",283,50.5
9,Mais envergadura,4,"4"" ou mais",505,55.0


In [25]:
# Conferência: mesmas faixas do gráfico idade_vs_envergadura.png, com pd.cut sobre as datas em Pandas
datas_ev = pd.read_csv("../data/processed/eventos.csv")[["Event_URL", "Event_Date"]]
f = df.merge(lutadores, on="Fighter_URL").merge(datas_ev, on="Event_URL")
f["idade"] = (pd.to_datetime(f["Event_Date"]) - pd.to_datetime(f["DOB"])).dt.days / 365.25
f["env"] = (f["Reach_cm"] / 2.54).round(1)
v = f[f["Resultado"] == "W"].set_index("Fight_URL")[["idade", "env"]]
p = f[f["Resultado"] == "L"].set_index("Fight_URL")[["idade", "env"]]
d = v.join(p, lsuffix="_v", rsuffix="_p", how="inner")
dif_idade, dif_env = d["idade_v"] - d["idade_p"], (d["env_v"] - d["env_p"]).round(1)
dif_env = dif_env[dif_env != 0]
paineis = {
    "Mais novo": ((dif_idade < 0).groupby(pd.cut(dif_idade.abs(), [0, 2, 5, 8, 99], labels=[1, 2, 3, 4]), observed=True), dif_idade < 0),
    "Mais envergadura": ((dif_env > 0).groupby(pd.cut(dif_env.abs(), [0, 1, 2, 3, 99], labels=[1, 2, 3, 4]), observed=True), dif_env > 0),
}
for vantagem, (por_faixa, todas) in paineis.items():
    esperado = por_faixa.agg(["size", "mean"])
    esperado.index = esperado.index.astype(int)  # faixas do pd.cut (categóricas) -> 1..4, como a coluna ordem
    esperado.loc[0] = [len(todas), todas.mean()]
    esperado = esperado.sort_index()
    sql = fisico[fisico["vantagem"] == vantagem].set_index("ordem").sort_index()
    assert list(sql.index) == list(esperado.index)
    assert list(sql["lutas"]) == list(esperado["size"].astype(int))
    assert list(sql["pct_vantagem_venceu"]) == list((esperado["mean"] * 100).round(1))
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem

## 14. Consultas que só aparecem aqui

Perguntas que seriam trabalhosas em Pandas e ficam naturais com funções de janela.

### Destaque de cada evento

`ROW_NUMBER() OVER (PARTITION BY evento)` escolhe quem mais acertou golpes significativos numa luta, em cada evento.

In [26]:
mostrar("destaque_por_evento")

-- Em cada evento, quem acertou mais golpes significativos numa única luta?
-- ROW_NUMBER() por evento escolhe um só por evento; mostra os 10 maiores desses recordes.
WITH ranqueado AS (
    SELECT
        e.nome        AS evento,
        d.lutador,
        d.sig_acertados,
        ROW_NUMBER() OVER (PARTITION BY l.evento_id ORDER BY d.sig_acertados DESC, d.lutador) AS n
    FROM desempenho AS d
    JOIN lutas   AS l USING (luta_id)
    JOIN eventos AS e USING (evento_id)
)
SELECT evento, lutador, sig_acertados
FROM ranqueado
WHERE n = 1
ORDER BY sig_acertados DESC
LIMIT 10;



,evento,lutador,sig_acertados
0,UFC Fight Night: Du Plessis vs. Usman,Tommy McMillen,252
1,UFC Fight Night: Vettori vs. Cannonier,Jared Cannonier,241
2,UFC 299: O'Malley vs. Vera 2,Sean O'Malley,230
3,UFC Fight Night: Barboza vs. Murphy,Lerone Murphy,220
4,UFC 317: Topuria vs. Oliveira,Brandon Royval,215
5,UFC 296: Edwards vs. Covington,Karol Rosa,204
6,UFC 318: Holloway vs. Poirier 3,Max Holloway,198
7,UFC 300: Pereira vs. Hill,King Green,186
8,UFC 302: Makhachev vs. Poirier,Sean Strickland,182
9,UFC 331: Van vs. Pantoja 2,Joshua Van,181


### Maior sequência de vitórias

Problema clássico de *gaps and islands*: com as lutas de cada lutador em ordem de data, dentro de uma sequência de resultados iguais a diferença entre o número da luta na carreira e o número da luta entre as de mesmo resultado é constante — e vira o identificador da sequência.

In [27]:
mostrar("sequencia_vitorias")

-- Maior sequência de vitórias seguidas de cada lutador no período ("gaps and islands"):
-- a diferença entre dois ROW_NUMBER() é constante dentro de uma sequência de mesmo resultado.
-- As lutas de cada lutador são ordenadas pela data do evento.
WITH ordenado AS (
    SELECT
        d.lutador,
        d.resultado,
        ROW_NUMBER() OVER (PARTITION BY d.lutador ORDER BY e.data)
          - ROW_NUMBER() OVER (PARTITION BY d.lutador, d.resultado ORDER BY e.data) AS grupo
    FROM desempenho AS d
    JOIN lutas   AS l USING (luta_id)
    JOIN eventos AS e USING (evento_id)
),
sequencias AS (
    SELECT lutador, COUNT(*) AS vitorias_seguidas
    FROM ordenado
    WHERE resultado = 'W'
    GROUP BY lutador, grupo
)
SELECT lutador, MAX(vitorias_seguidas) AS maior_sequencia
FROM sequencias
GROUP BY lutador
ORDER BY maior_sequencia DESC, lutador
LIMIT 10;



,lutador,maior_sequencia
0,Joshua Van,8
1,Carlos Ulberg,7
2,Ailin Perez,6
3,Farid Basharat,6
4,Joaquin Buckley,6
5,Melquizael Costa,6
6,Natalia Silva,6
7,Navajo Stirling,6
8,Quillan Salkilld,6
9,Steve Garcia,6


### Como as lutas terminam, ano a ano

Usa a data de cada evento (`strftime('%Y', data)`). 2023 e o ano corrente são parciais: o recorte começa em abril de 2023.

In [28]:
por_ano = mostrar("lutas_por_ano")
por_ano

-- Como as lutas terminam a cada ano do período (strftime extrai o ano da data do evento).
-- 2023 e o ano corrente são parciais: o recorte começa em abril de 2023.
SELECT
    strftime('%Y', e.data)                                                          AS ano,
    COUNT(DISTINCT e.evento_id)                                                     AS eventos,
    COUNT(*)                                                                        AS lutas,
    ROUND(100.0 * SUM(l.metodo LIKE '%KO/TKO%' OR l.metodo LIKE 'TKO - Doctor%') / COUNT(*), 1) AS nocaute_pct,
    ROUND(100.0 * SUM(l.metodo LIKE '%Submission%') / COUNT(*), 1)                  AS finalizacao_pct,
    ROUND(100.0 * SUM(l.metodo LIKE 'Decision%') / COUNT(*), 1)                     AS decisao_pct
FROM lutas AS l
JOIN eventos AS e USING (evento_id)
GROUP BY ano
ORDER BY ano;



,ano,eventos,lutas,nocaute_pct,finalizacao_pct,decisao_pct
0,2023,33,397,32.7,16.9,47.6
1,2024,42,517,28.4,16.1,54.4
2,2025,42,520,32.5,17.5,49.2
3,2026,32,403,37.5,17.9,44.2


In [29]:
# Conferência: mesma contagem de lutas por ano feita em Pandas, juntando as datas de eventos.csv
datas = pd.read_csv("../data/processed/eventos.csv").set_index("Event_URL")["Event_Date"]
ano = df.drop_duplicates("Fight_URL")["Event_URL"].map(datas).str[:4]
assert por_ano.set_index("ano")["lutas"].to_dict() == ano.value_counts().to_dict()
print("OK: SQL e Pandas batem")

OK: SQL e Pandas batem
